# 🛍️ Customer Segmentation Using RFM Analysis
### E-Commerce Behavioral Analytics | UCI Online Retail Dataset

**Objective:** Segment customers based on purchasing behavior to identify 
high-value customers, at-risk churners, and growth opportunities.

**Tools:** Python · pandas · matplotlib · seaborn · plotly

**Dataset:** UCI Online Retail Dataset — 541,909 transactions from a UK-based 
online retailer (2010–2011)

In [ ]:
!pip install pandas numpy matplotlib seaborn plotly openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("✅ Libraries loaded successfully")

In [ ]:
# Load dataset
df = pd.read_excel('C:\\Project\\customer-segmentation-rfm\\data\\Online Retail.xlsx', engine='openpyxl')

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 1. Exploratory Data Analysis (EDA)

Before cleaning, let's understand what we're working with.

In [4]:
print("=== BASIC INFO ===")
print(f"Total rows     : {df.shape[0]:,}")
print(f"Total columns  : {df.shape[1]}")
print(f"Date range     : {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")
print(f"Unique customers: {df['CustomerID'].nunique():,}")
print(f"Unique products : {df['StockCode'].nunique():,}")
print(f"Countries       : {df['Country'].nunique()}")

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DATA TYPES ===")
print(df.dtypes)

=== BASIC INFO ===
Total rows     : 541,909
Total columns  : 8
Date range     : 2010-12-01 08:26:00 → 2011-12-09 12:50:00
Unique customers: 4,372
Unique products : 4,070
Countries       : 38

=== MISSING VALUES ===
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

=== DATA TYPES ===
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object


## 2. Data Cleaning

Issues to address:
- Remove rows with missing `CustomerID` (can't do RFM without it)
- Remove cancelled orders (InvoiceNo starting with 'C')
- Remove rows where Quantity or UnitPrice ≤ 0
- Keep only United Kingdom for focused analysis

In [5]:
df_clean = df.copy()

# Drop missing CustomerID
df_clean = df_clean.dropna(subset=['CustomerID'])

# Remove cancellations
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# Remove negative/zero quantity and price
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# Focus on UK customers
df_clean = df_clean[df_clean['Country'] == 'United Kingdom']

# Create TotalPrice column
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Ensure correct date type
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

print(f"Rows before cleaning : {df.shape[0]:,}")
print(f"Rows after cleaning  : {df_clean.shape[0]:,}")
print(f"Rows removed         : {df.shape[0] - df_clean.shape[0]:,}")
print(f"\nCustomers remaining  : {df_clean['CustomerID'].nunique():,}")

Rows before cleaning : 541,909
Rows after cleaning  : 354,321
Rows removed         : 187,588

Customers remaining  : 3,920
